# AURA — Drift Monitor Demo

`DriftMonitor` keeps an append-only JSONL log of predictions and
confirmations, maintains a 2×2 confusion matrix in memory, and
surfaces a `DriftSignal` whose `status` flips to `WARNING` when the
false-positive rate crosses a configured threshold. State is
reconstructed from the log on construction, so the monitor survives
process restarts.

This notebook walks through:

1. Creating a monitor, wiring it into `PhishingDetector`
2. Running a few predictions and recording confirmations
3. Crossing the FPR threshold and reading the `WARNING` signal
4. Restarting — constructing a fresh monitor over the same log —
   and confirming the confusion matrix replays byte-for-byte.

All predictions are synthetic; no network calls.

## 1. Setup

Resolve the repository root, import the public API, and pick a
scratch directory for the JSONL log. We deliberately delete any
pre-existing log so the demo starts from a clean slate.

In [1]:
import logging
import os
import sys
import tempfile
from pathlib import Path

import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('aura.inference').setLevel(logging.INFO)

repo_root = Path.cwd()
for _ in range(4):
    if (repo_root / 'inference' / '__init__.py').exists():
        break
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

os.environ.setdefault('AURA_MODELS_DIR', str((repo_root / 'models').resolve()))

from inference import (
    DriftMonitor, DriftSignal, DriftStatus, ModelRegistry, PhishingDetector,
)

log_dir = Path(tempfile.mkdtemp(prefix='aura-drift-'))
log_path = log_dir / 'drift.jsonl'
print('drift log path:', log_path)

drift log path: C:\Users\Administrator\AppData\Local\Temp\aura-drift-pw0edl1u\drift.jsonl


## 2. Wire a monitor into the detector

Passing `drift_monitor=` to `PhishingDetector.from_paths` means every
call to `predict` / `predict_batch` automatically calls
`record_prediction(prediction_id, predicted_label, predicted_probability, ...)`.
You only need to call `record_confirmation(...)` when a ground-truth
label later becomes available.

We set `fpr_threshold=0.20` so the demo crosses it with a small
number of confirmations.

In [2]:
monitor = DriftMonitor(log_path, fpr_threshold=0.20)

registry = ModelRegistry(Path(os.environ['AURA_MODELS_DIR']))
version = registry.active_version() or registry.latest_version()
paths = registry.paths_for(version)

detector = PhishingDetector.from_paths(
    model_path=paths['model'],
    subject_vectorizer_path=paths['subject_vectorizer'],
    body_vectorizer_path=paths['body_vectorizer'],
    calibrator_path=paths.get('calibrator'),
    drift_monitor=monitor,
)
detector.version = version
print('version   :', detector.version)
print('monitor   :', monitor.log_path.name, '(fpr_threshold =', monitor.fpr_threshold, ')')

version   : v1_0
monitor   : drift.jsonl (fpr_threshold = 0.2 )


## 3. Run predictions

Two phishing-looking emails and four benign emails. Predictions are
automatically recorded; the `prediction_id` each result carries is
what we'll feed into `record_confirmation` below.

In [3]:
emails = [
    # phishy
    {'sender': '"PayPal" <service@paypa1-alerts.com>',
     'subject': 'URGENT: verify now!!!',
     'body': 'Verify at http://paypa1-alerts.com/verify within 24h.',
     'true_label': 1},
    {'sender': '"DHL" <tracking@dhl-delivery-notice.info>',
     'subject': 'Delivery failed — reschedule',
     'body': 'Confirm address at http://dhl-delivery-notice.info/track?id=12',
     'true_label': 1},
    # benign
    {'sender': '"Jane" <jane@gmail.com>',
     'subject': 'Lunch on Saturday?',
     'body': 'Free around noon? Thinking of that ramen place on 5th.',
     'true_label': 0},
    {'sender': '"GitHub" <noreply@github.com>',
     'subject': 'Weekly digest',
     'body': 'Top repositories this week. Visit github.com/trending.',
     'true_label': 0},
    {'sender': '"ACM" <newsletter@acm.org>',
     'subject': 'ACM TechNews — April edition',
     'body': 'Advances in compilers, a Unix retrospective, upcoming deadlines.',
     'true_label': 0},
    {'sender': '"Mom" <mom@familymail.net>',
     'subject': 'Recipe for that soup',
     'body': 'Soak lentils overnight, simmer with carrots and celery.',
     'true_label': 0},
]

results = [
    detector.predict(e['sender'], e['subject'], e['body'], threshold=0.5)
    for e in emails
]

pd.DataFrame([
    {'prediction_id': r.prediction_id[:8] + '…',
     'predicted_label': r.predicted_label,
     'phish_prob': round(r.phishing_probability, 4),
     'true_label': e['true_label']}
    for r, e in zip(results, emails)
])

C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MLPClassifier was fitted with feature names
  warnings.warn(


,prediction_id,predicted_label,phish_prob,true_label
0,56cbd48c…,1,0.9987,1
1,76f87313…,1,0.9954,1
2,4be8e783…,0,0.2634,0
3,ec86024c…,1,0.9865,0
4,cd954624…,0,0.0891,0
5,54ed0f83…,1,0.9763,0


## 4. Record confirmations and cross the FPR threshold

To deliberately cross the FPR threshold we fabricate a false-positive
outcome on two of the benign emails — we tell the monitor the model
flagged them even when it did not. In practice the monitor records
whatever `predicted_label` came out of `predict()`; here we force
them up via `record_prediction` so the demo reliably warns.

In [4]:
# Inject two synthetic false-positive predictions: predicted 1, truth 0.
import uuid
fp_ids = []
for _ in range(2):
    pid = str(uuid.uuid4())
    monitor.record_prediction(
        prediction_id=pid,
        predicted_label=1,
        predicted_probability=0.9,
        model_version=detector.version,
    )
    monitor.record_confirmation(prediction_id=pid, confirmed_label=0)
    fp_ids.append(pid)

# Confirm the genuine ones so the confusion matrix has the other three cells populated.
for r, e in zip(results, emails):
    monitor.record_confirmation(
        prediction_id=r.prediction_id,
        confirmed_label=e['true_label'],
    )

print('confusion matrix:', monitor.confusion_matrix())
print('false_positive_rate:', round(monitor.false_positive_rate(), 4))
signal = monitor.drift_signal()
print('signal status :', signal.status.value)
print('signal message:', signal.message)

confusion matrix: {'tp': 2, 'tn': 2, 'fp': 4, 'fn': 0}
false_positive_rate: 0.6667
signal status : WARNING
signal message: FPR 0.6667 exceeds threshold 0.2000 — model may need retraining


## 5. Restart — replay from the JSONL log

A new `DriftMonitor` constructed over the same log reads every
record from disk and rebuilds the in-memory state. The confusion
matrix must match byte-for-byte what the first instance held.

In [5]:
reloaded = DriftMonitor(log_path, fpr_threshold=0.20)

print('original cm:', monitor.confusion_matrix())
print('reloaded cm:', reloaded.confusion_matrix())
print('match      :', monitor.confusion_matrix() == reloaded.confusion_matrix())
print('reloaded signal status:', reloaded.drift_signal().status.value)

original cm: {'tp': 2, 'tn': 2, 'fp': 4, 'fn': 0}
reloaded cm: {'tp': 2, 'tn': 2, 'fp': 4, 'fn': 0}
match      : True
reloaded signal status: WARNING


### Raw JSONL tail

For curiosity: every record written to disk has a `type` field
(`prediction` or `confirmation`) plus the minimal fields the monitor
needs to rebuild state. Nothing else is written — the log is stable
to tail from an external dashboard.

In [6]:
lines = log_path.read_text(encoding='utf-8').splitlines()
print(f'{len(lines)} records on disk — last 5:')
for line in lines[-5:]:
    print(line)

16 records on disk — last 5:
{"type": "confirmation", "prediction_id": "76f87313-73e5-41e9-a0a1-982eb0ec9bfb", "confirmed_label": 1, "timestamp": "2026-04-18T05:56:40.440306+00:00"}
{"type": "confirmation", "prediction_id": "4be8e783-5cc3-40cd-9153-fbc518168b4a", "confirmed_label": 0, "timestamp": "2026-04-18T05:56:40.440306+00:00"}
{"type": "confirmation", "prediction_id": "ec86024c-a2b7-4bd1-bd24-bcd06873f274", "confirmed_label": 0, "timestamp": "2026-04-18T05:56:40.441306+00:00"}
{"type": "confirmation", "prediction_id": "cd954624-4921-4eba-b458-d685668cc060", "confirmed_label": 0, "timestamp": "2026-04-18T05:56:40.441306+00:00"}
{"type": "confirmation", "prediction_id": "54ed0f83-57e0-486b-a258-6aa2d5a502e7", "confirmed_label": 0, "timestamp": "2026-04-18T05:56:40.442306+00:00"}
